# APIs : enrichir des noms du corpus CAMille avec Wikidata

Dans ce notebook, nous interrogeons l'[API de Wikidata](https://www.wikidata.org/) pour trouver, pour des personnes citées dans le corpus CAMille, leur description, leur année de naissance et leur année de décès. Le résultat est sauvegardé dans un fichier csv.

## Imports

In [ ]:
import json
import os
import time

import pandas as pd
import requests

## Utiliser [Wikidata](https://www.wikidata.org/) pour chercher une personne

Wikidata demande que chaque requête contienne un `User-Agent` qui identifie le programme.

In [ ]:
headers = {'User-Agent': 'TAC-TP1-api/1.0 (cours STIC-B545, ULB)'}
api_url = "https://www.wikidata.org/w/api.php"

name = "René Hislaire"

"""Chercher les identifiants Wikidata correspondant au nom"""
params = {'action': 'wbsearchentities', 'search': name, 'language': 'fr', 'format': 'json', 'limit': 5}
resp = requests.get(api_url, params=params, headers=headers)
try:
    json_list = resp.json()['search']
except (ValueError, KeyError): # la réponse n'est pas du JSON attendu, la requête est probablement bloquée
    print(resp.text[:300])
    json_list = []
for item in json_list:
    print(f"{item['id']} | {item.get('label', '')} | {item.get('description', '')}")

## Récupérer les informations d'une entité

On prend le premier résultat. Vérifiez toujours la description : un nom courant peut désigner plusieurs personnes.

In [ ]:
def annee(claims, propriete):
    """Année d'une propriété de date (P569 = naissance, P570 = décès), ou chaîne vide"""
    try:
        temps = claims[propriete][0]['mainsnak']['datavalue']['value']['time']
    except (KeyError, IndexError, TypeError):
        return ''
    return temps[1:5]  # par exemple '+1891-00-00T00:00:00Z' -> '1891'

if json_list:
    qid = json_list[0]['id']
    params = {'action': 'wbgetentities', 'ids': qid, 'props': 'labels|descriptions|claims', 'languages': 'fr', 'format': 'json'}
    resp = requests.get(api_url, params=params, headers=headers)
    entity = resp.json()['entities'][qid]
    print('Identifiant :', qid)
    print('Description :', entity.get('descriptions', {}).get('fr', {}).get('value', ''))
    print('Naissance   :', annee(entity.get('claims', {}), 'P569'))
    print('Décès       :', annee(entity.get('claims', {}), 'P570'))

## Enrichir plusieurs noms du corpus et sauvegarder un fichier csv

In [ ]:
# Noms à enrichir : à remplacer par les noms relevés dans le corpus
names = ["René Hislaire", "Marguerite Van de Wiele", "Alice Bron", "Louise Van den Plas"]

rows = []
for name in names:
    params = {'action': 'wbsearchentities', 'search': name, 'language': 'fr', 'format': 'json', 'limit': 5}
    resp = requests.get(api_url, params=params, headers=headers)
    results = resp.json().get('search', [])
    if not results:
        rows.append([name, '', '', '', '']) 
        print(f"{name} : aucun résultat")
        continue
    qid = results[0]['id']
    params = {'action': 'wbgetentities', 'ids': qid, 'props': 'labels|descriptions|claims', 'languages': 'fr', 'format': 'json'}
    entity = requests.get(api_url, params=params, headers=headers).json()['entities'][qid]
    description = entity.get('descriptions', {}).get('fr', {}).get('value', '')
    rows.append([name, qid, description, annee(entity.get('claims', {}), 'P569'), annee(entity.get('claims', {}), 'P570')])
    time.sleep(1)  # pause entre deux requêtes

df = pd.DataFrame(rows, columns=['nom', 'qid', 'description', 'naissance', 'décès'])
os.makedirs("../data", exist_ok=True)
df.to_csv(f"../data/camille_wikidata_{time.strftime('%Y%m%d')}.csv", index=False)
df

## Pour en savoir plus

- Les API Rest: https://towardsdatascience.com/introduction-to-rest-apis-90b5d9676004
- La librairie [`requests`](https://requests.readthedocs.io/en/latest/)
- Documentation de l'API Wikidata : https://www.wikidata.org/w/api.php et https://www.wikidata.org/wiki/Wikidata:Data_access